# Colab training notebook

Notebook nay dung de train News-Article-Recommendation-System voi MINDsmall tren Google Colab.

Truoc khi chay: vao `Runtime` -> `Change runtime type` -> chon `GPU` neu co.


In [ ]:
# Check GPU runtime. Neu khong co GPU, training van chay CPU nhung se cham hon.
import subprocess

try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as exc:
    print("No GPU detected. Use Runtime > Change runtime type > GPU if available.")


## 1. Mount Google Drive

Dat MINDsmall trong Google Drive. Folder co the la folder chua truc tiep `news.tsv`/`behaviors.tsv`, hoac folder cha co `MINDsmall_train`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Clone or update repository


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ThienTan142/News-Article-Recommendation-System.git"
BRANCH = "codex/mindsmall-pipeline-cleanup"
PROJECT_DIR = Path("/content/News-Article-Recommendation-System")

def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

if PROJECT_DIR.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR])

print("Project dir:", PROJECT_DIR)


## 3. Install dependencies


In [ ]:
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=PROJECT_DIR)


## 4. Configure MINDsmall path

Sua `MIND_DIR` cho dung vi tri dataset cua ban tren Google Drive.

Vi du hop le:

- `/content/drive/MyDrive/MINDsmall`
- `/content/drive/MyDrive/Project/data/MINDsmall`
- `/content/drive/MyDrive/MINDsmall/MINDsmall_train`


In [ ]:
MIND_DIR = "/content/drive/MyDrive/MINDsmall"

sys.path.insert(0, str(PROJECT_DIR))
from src.mind_dataset import resolve_mindsmall_paths

paths = resolve_mindsmall_paths(MIND_DIR)
print("Resolved train dir:", paths.root)
print("News:", paths.news_path)
print("Behaviors:", paths.behaviors_path)


## 5. Build precompute artifacts

Cell `precompute_news.py` co the mat vai phut vi phai encode toan bo news bang SentenceTransformer.


In [ ]:
run([sys.executable, "scripts/precompute_news.py", "--mind-dir", MIND_DIR], cwd=PROJECT_DIR)
run([sys.executable, "scripts/precompute_user_history.py", "--mind-dir", MIND_DIR], cwd=PROJECT_DIR)


## 6. Build CTR dataset


In [ ]:
run([sys.executable, "scripts/build_ctr_dataset.py", "--mind-dir", MIND_DIR, "--neg-ratio", "4"], cwd=PROJECT_DIR)


## 7. Train CTR reranker

Mac dinh duoi day dung cau hinh full hon local smoke test. Neu Colab het RAM/VRAM, giam `BATCH_SIZE` xuong `512` hoac `256`.


In [ ]:
import torch

TRAIN_MAX_ROWS = 150000
EPOCHS = 8
BATCH_SIZE = 1024

env = os.environ.copy()
env["NEWS_REC_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", env["NEWS_REC_DEVICE"])

run([
    sys.executable,
    "-m",
    "src.train",
    "--max-rows",
    TRAIN_MAX_ROWS,
    "--epochs",
    EPOCHS,
    "--batch-size",
    BATCH_SIZE,
], cwd=PROJECT_DIR, env=env)


## 8. Run recommendation


In [ ]:
USER_ID = "U8125"
run([sys.executable, "-m", "src.run_recommend_cli", "--user", USER_ID, "--topk", "10", "--json"], cwd=PROJECT_DIR)


## 9. Copy artifacts back to Google Drive

Nhung file nay khong nen commit vao Git. Luu trong Drive de tai ve dung local/demo.


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/news-rec-artifacts")
ARTIFACTS = [
    "models/ctr_model.pt",
    "data/precompute/news_embeddings.npy",
    "data/precompute/news_metadata.csv",
    "data/precompute/user_history.json",
    "data/precompute/ctr_dataset.csv",
    "data/precompute/manifest.json",
]

for rel_path in ARTIFACTS:
    source = PROJECT_DIR / rel_path
    target = OUTPUT_DIR / rel_path
    if source.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        print("Copied", source, "->", target)
    else:
        print("Missing artifact, skipped:", source)
